# Ejercicio 2 — Regresión Lineal con Scikit-Learn
## Practica Final: Estadística para Data Science

**Autor:** Alejandro Pujana Quintero  
**Proposito:** Cuaderno de trabajo paralelo a `ejercicio2_inferencia.py`.  
Documenta la logica, decisiones tecnicas e interpretacion econometrica del modelo.

---

### Objetivo del ejercicio
Entrenar un modelo de regresion lineal multiple con sklearn para predecir `loan_amount`  
y evaluar su capacidad predictiva e inferencia estadistica sobre los datos de test.

### Conexion con el Ejercicio 1
Los hallazgos del EDA determinaron directamente las decisiones de este ejercicio:

| Hallazgo EDA | Decision en Ej.2 |
|---|---|
| `payment_to_income_ratio` r=1.000 | Excluida — multicolinealidad perfecta |
| `savings_assets` curtosis=57.8 | StandardScaler mitiga impacto de outliers extremos |
| `occupation_status`, `product_type`, `loan_intent` son str | OHE con drop_first=True |
| `loan_status` es resultado post-decision | Excluida — data leakage |
| Top-3 correlaciones con target | Confirman los predictores mas importantes del modelo |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import statsmodels.api as sm

np.random.seed(42)
sns.set_theme(style='whitegrid', palette='muted')

BASE_DIR  = Path('..') / 'practica_final_Pujana_Quintero_Alejandro'
DATA_PATH = BASE_DIR / 'data' / 'Loan_approval_data_2025.csv'

df = pd.read_csv(DATA_PATH)
print(f'Dataset: {df.shape[0]:,} filas x {df.shape[1]} columnas')

## Preprocesamiento

### 1. Eliminacion de columnas

**`customer_id`**: identificador administrativo sin valor predictivo.  
**`loan_status`**: data leakage — es el resultado de la decision de credito. En produccion no existiria al predecir.  
**`payment_to_income_ratio`**: multicolinealidad perfecta (r=1.000) con `loan_to_income_ratio`.  
Con ambas incluidas, XtX es singular y el OLS no tiene solucion unica.

### 2. One-Hot Encoding con drop_first=True

Las variables categoricas no tienen orden numerico — no podemos pasarlas directamente al modelo.  
OHE crea columnas binarias (0/1) por cada categoria.

`drop_first=True` elimina la primera categoria de cada variable para evitar la **trampa de variables dummy**:  
si incluyes todas las categorias, son perfectamente colineales (suman 1 siempre), violando OLS.

Categorias de referencia (interpretacion ceteris paribus):
- `occupation_status` -> **Employed** (base)
- `product_type`      -> **Credit Card** (base)
- `loan_intent`       -> **Business** (base)

### 3. StandardScaler — por que es necesario para el grafico de coeficientes

En OLS, los coeficientes no estandarizados se expresan en las unidades de cada variable:  
- `annual_income`: beta en USD/USD (escala ~50,000)  
- `credit_score`: beta en USD/punto (escala ~300-850)  
- `age`: beta en USD/ano (escala 18-70)

No son comparables entre si. Al estandarizar (media=0, std=1), los coeficientes expresan  
el cambio en loan_amount por **1 desviacion tipica** de cada variable — ahi si son comparables.

> CRITICO: el scaler se ajusta SOLO sobre X_train con `.fit_transform()`.  
> Sobre X_test se usa solo `.transform()`.  
> Si ajustaras sobre todo el dataset antes del split, el modelo 'veria' la distribucion  
> del test set antes de predecir — otro tipo de data leakage.

### 4. Split 80/20

80% train (40,000 obs) para ajustar los coeficientes.  
20% test (10,000 obs) para evaluar como generaliza a datos nuevos.  
`random_state=42` garantiza reproducibilidad — la misma particion en cualquier maquina.

In [ ]:
TARGET    = 'loan_amount'
DROP_COLS = ['customer_id', 'loan_status', 'payment_to_income_ratio']
CAT_COLS  = ['occupation_status', 'product_type', 'loan_intent']

df_clean = df.drop(columns=DROP_COLS)
df_ohe   = pd.get_dummies(df_clean, columns=CAT_COLS, drop_first=True, dtype=int)

X = df_ohe.drop(columns=[TARGET])
y = df_ohe[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler     = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Features post-OHE: {X.shape[1]}')
print(f'Train: {X_train.shape[0]:,} | Test: {X_test.shape[0]:,}')
print(f'\nColumnas OHE generadas:')
ohe_cols = [c for c in X.columns if any(c.startswith(cat+'_') for cat in CAT_COLS)]
for c in ohe_cols:
    print(f'  {c}')

## Modelo A — Regresion Lineal (sklearn)

### Por que comparar train Y test

En econometria clasica (STATA `reg`) el foco es la **inferencia**: signo, magnitud y  
significancia de los betas sobre los datos disponibles.  

En ML supervisado el foco es la **generalizacion**: que tan bien predice el modelo  
sobre datos que **nunca vio** durante el entrenamiento.  

Para diagnosticar overfitting/underfitting necesitas los dos:

| Patron | R2_train | R2_test | Diagnostico |
|---|---|---|---|
| R2_train >> R2_test | Alto | Bajo | Overfitting: memorizo el ruido |
| R2_train ≈ R2_test bajos | Bajo | Bajo | Underfitting: modelo insuficiente |
| R2_train ≈ R2_test altos | Alto | Alto | Buen ajuste: generaliza bien |

### Interpretacion de las metricas

**MAE** (Mean Absolute Error): error promedio en USD. Interpretacion directa:  
en promedio el modelo se equivoca en $X al predecir el monto del prestamo.

**RMSE** (Root Mean Squared Error): penaliza errores grandes por el cuadrado.  
Si RMSE >> MAE, hay predicciones muy malas en algunos casos especificos.  
Aqui RMSE ($11,207) > MAE ($7,561) — confirma la presencia de outliers en los residuos.

**R²**: proporcion de la varianza de loan_amount explicada por el modelo.  
R²=0.817 significa que el 81.7% de la variabilidad del monto esta explicada  
por las features incluidas. El 18.3% restante es variabilidad no capturada (ruido).

In [ ]:
modelo = LinearRegression()
modelo.fit(X_train_sc, y_train)

y_pred_train = modelo.predict(X_train_sc)
y_pred_test  = modelo.predict(X_test_sc)

resultados = pd.DataFrame({
    'Metrica': ['MAE', 'RMSE', 'R2'],
    'Train': [
        mean_absolute_error(y_train, y_pred_train),
        np.sqrt(mean_squared_error(y_train, y_pred_train)),
        r2_score(y_train, y_pred_train)
    ],
    'Test': [
        mean_absolute_error(y_test, y_pred_test),
        np.sqrt(mean_squared_error(y_test, y_pred_test)),
        r2_score(y_test, y_pred_test)
    ]
}).set_index('Metrica')

resultados['Diferencia'] = (resultados['Train'] - resultados['Test']).abs()
print(resultados.round(4))

diff_r2 = r2_score(y_train, y_pred_train) - r2_score(y_test, y_pred_test)
print(f'\nDiagnostico: R2_train - R2_test = {diff_r2:.4f}')
if abs(diff_r2) < 0.02:
    print('Sin overfitting detectable. El modelo generaliza correctamente.')

### Interpretacion de las metricas — resultados reales

**MAE_test = $7,561 USD**  
En promedio el modelo se equivoca en $7,561 al predecir el monto del prestamo.  
Sobre un rango de $500–$100,000, esto representa un error relativo del 7.6% sobre el maximo.  
Sobre la mediana ($26,100), el error es del 29% — considerable pero esperable  
para un modelo lineal con alta variabilidad en el target (std=$26,116).

**RMSE_test = $11,207 vs MAE_test = $7,561**  
La diferencia (RMSE/MAE = 1.48) indica la presencia de errores grandes en algunos casos.  
En STATA estariamos viendo residuos con alta curtosis — confirmado: curtosis residuos = 7.67.  
El modelo falla sistematicamente en los extremos del rango de prestamos.

**R2_train = 0.810 vs R2_test = 0.817**  
Diferencia de -0.007 (R2_test > R2_train).  
**Sin overfitting**. Es normal que R2_test sea ligeramente mayor que R2_train  
cuando el test set, por azar, tiene una distribucion ligeramente mas predecible.  
El modelo generaliza correctamente — no memorizo ruido del training.

**Nota sobre loan_to_income_ratio**  
El R2=0.81 esta fuertemente influenciado por `loan_to_income_ratio` (r=0.66 con target,  
y coeficiente estandarizado $19,231 — el mas alto con diferencia).  
Esta variable = loan_amount/annual_income, por lo que contiene el target en el numerador.  
En un sistema real habria que evaluar si este ratio existe *antes* de fijar el monto.  
Lo documentamos como limitacion metodologica, no como error del modelo.

## Analisis de Residuos

### Que esperamos ver en residuos de un OLS bien especificado

Los supuestos clasicos de OLS sobre los residuos (Gauss-Markov) son:
1. **Media cero**: E[e] = 0. Los errores se cancelan en promedio.
2. **Homocedasticidad**: Var(e) = σ² constante. La varianza del error no depende de X.
3. **No autocorrelacion**: Cov(ei, ej) = 0 para i≠j.
4. **Normalidad**: e ~ N(0, σ²) para inferencia estadistica valida.

### Que vemos realmente — heterocedasticidad

El grafico de residuos vs predichos muestra un **patron de abanico** (forma triangular):  
a medida que el valor predicho crece, la dispersion de los residuos tambien crece.  
Esto viola el supuesto 2 (homocedasticidad).

**Origen del patron**: viene de `loan_to_income_ratio` dominando la prediccion.  
El modelo predice en 'bandas' segun el ratio, generando las lineas diagonales visibles.  
Las predicciones para prestamos grandes (>$60,000) tienen errores sistematicamente mayores.

**Consecuencia estadistica**: los errores estandar del OLS clasico no son confiables.  
En STATA, la solucion seria usar `robust` (errores estandar de Huber-White).  
En el contexto de este ejercicio, lo documentamos sin corregir — el enunciado no lo requiere.

**Conexion con Ejercicio 1**: este patron era predecible.  
`savings_assets` tiene curtosis=57.8 y `derogatory_marks` 12.79% outliers.  
Decisiones que en la vida real modificarian la estrategia: transformacion logaritmica  
del target o de variables sesgadas, o usar un modelo mas flexible (Random Forest, XGBoost).

In [ ]:
residuos = y_test.values - y_pred_test

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(y_pred_test, residuos, alpha=0.3, s=8, color=sns.color_palette('muted')[0])
axes[0].axhline(0, color='crimson', linewidth=1.5, linestyle='--')
axes[0].set_xlabel('Valores Predichos (USD)')
axes[0].set_ylabel('Residuos (USD)')
axes[0].set_title('Residuos vs Valores Predichos\n(Heterocedasticidad visible = abanico)', fontweight='bold')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

axes[1].hist(residuos, bins=50, color=sns.color_palette('muted')[1], edgecolor='white', alpha=0.85)
axes[1].axvline(0, color='crimson', linewidth=1.5, linestyle='--', label='Residuo=0')
axes[1].axvline(np.mean(residuos), color='navy', linewidth=1.2, linestyle=':',
                label=f'Media: ${np.mean(residuos):,.0f}')
axes[1].set_xlabel('Residuo (USD)')
axes[1].set_title('Distribucion de Residuos\n(Leptocurtica: curtosis=7.67)', fontweight='bold')
axes[1].legend()
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

plt.tight_layout()
plt.show()

print(f'Media residuos  : ${np.mean(residuos):,.2f} (ideal: 0)')
print(f'Std residuos    : ${np.std(residuos):,.2f}')
print(f'Min / Max       : ${residuos.min():,.2f} / ${residuos.max():,.2f}')

## Grafico de Coeficientes Estandarizados

### Como leer este grafico

Cada barra representa el cambio en `loan_amount` cuando esa variable aumenta **1 desviacion tipica**,  
manteniendo todo lo demas constante (ceteris paribus).

**Barra azul**: efecto positivo — la variable aumenta el monto del prestamo.  
**Barra roja**: efecto negativo — la variable disminuye el monto del prestamo.

### Hallazgos principales

**`loan_to_income_ratio` ($19,231)**: domina la prediccion con diferencia.  
1 SD de aumento en este ratio implica $19,231 mas en el monto.  
Conecta con r=0.66 del Ejercicio 1 — era el predictor mas correlacionado.

**`annual_income` ($15,387)**: segundo predictor mas importante.  
1 SD de ingreso adicional (~$47,000 sobre la media) implica $15,387 mas de prestamo.  
Logica economica: el banco presta en funcion de la capacidad de repago.

**`age` ($1,015)**: tercer predictor.  
Cada SD de edad (~12 anos sobre la media) implica $1,015 mas de prestamo.  
Consistente con la hipotesis: solicitantes mayores tienen mayor patrimonio y estabilidad.

**`Producto: Line of Credit` (-$894)**: efecto negativo respecto a Credit Card.  
Las lineas de credito tienen montos sistematicamente menores que tarjetas de credito.  
Consistente con el boxplot del Ejercicio 1 donde Line of Credit tenia medianas mas bajas.

**Variables de loan_intent**: efectos pequenos y, como veremos en OLS, no significativos.  
El proposito del prestamo no determina el monto — el perfil financiero si.

In [ ]:
from matplotlib.patches import Patch

coefs    = pd.Series(modelo.coef_, index=X.columns)
top10    = coefs.abs().nlargest(10).index
coefs_t  = coefs[top10].sort_values()
colores  = ['#2196F3' if v > 0 else '#E53935' for v in coefs_t.values]

fig, ax = plt.subplots(figsize=(10, 7))
bars = ax.barh(range(len(coefs_t)), coefs_t.values, color=colores, edgecolor='white', alpha=0.85)
ax.set_yticks(range(len(coefs_t)))
ax.set_yticklabels(coefs_t.index, fontsize=9)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Coeficiente estandarizado (USD por 1 SD)')
ax.set_title('Top-10 Coeficientes — Regresion Lineal Estandarizada', fontweight='bold')
ax.legend(handles=[
    Patch(facecolor='#2196F3', label='Efecto positivo'),
    Patch(facecolor='#E53935', label='Efecto negativo')
], fontsize=8)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.tight_layout()
plt.show()

## Summary OLS — Statsmodels (inferencia estadistica)

### Por que usamos statsmodels ademas de sklearn

Sklearn optimiza para **prediccion** — minimiza MSE, devuelve coeficientes y metricas.  
Statsmodels hace **inferencia estadistica** — devuelve la distribucion de los coeficientes:  
errores estandar, t-statistics, p-values e intervalos de confianza.

Ambos dan los **mismos betas** — es el mismo algoritmo OLS.  
La diferencia es la informacion adicional que statsmodels calcula sobre cada coeficiente.

### Como leer la tabla (equivalente a `reg` en STATA)

| Columna | Que mide | Interpretacion |
|---|---|---|
| `coef` | Beta estimado | Cambio en loan_amount (USD) por 1 unidad de la variable, ceteris paribus |
| `std err` | Error estandar | Incertidumbre en la estimacion del beta |
| `t` | t-statistic = coef/std_err | Si abs(t) > 1.96 -> significativo al 5% |
| `P>abs(t)` | p-value | < 0.05 -> rechaza H0: beta=0 |
| `[0.025]` | IC 95% inferior | El beta real esta aqui con 95% de confianza |
| `[0.975]` | IC 95% superior | El beta real esta aqui con 95% de confianza |

### Hallazgos clave del summary

**Solo 6 de 22 variables son significativas al 5%**:
- `loan_to_income_ratio`: beta=$41,243 (t=297) — el predictor absolutamente dominante
- `annual_income`: beta=$0.47 por USD de ingreso (t=127) — cada dolar de ingreso adicional se traduce en $0.47 de mayor prestamo
- `age`: beta=$91.21 por ano (t=11) — cada ano adicional implica $91 mas de prestamo
- `years_employed`: beta=$34.52 por ano (t=3.4)
- `occupation_status_Self-Employed`: beta=$840 vs Employed (t=5.6)
- `product_type_Line of Credit`: beta=-$2,239 vs Credit Card (t=-4.2)

**`credit_score` no es significativo (p=0.525)** — hallazgo contraintuitivo.  
El score crediticio no determina *cuanto* presta el banco en este dataset, solo *si* aprueba  
(loan_status, excluido). Su efecto esta mediado por la tasa de interés (r=-0.49 del Ej.1):  
mejor score -> menor tasa -> pero el monto lo determina el ingreso, no el score.

**`loan_intent` — ninguna categoria es significativa**.  
Confirmacion del hallazgo del Ejercicio 1: las medianas de loan_amount eran similares  
entre todas las categorias de intencion de prestamo.

**Condition Number = 2.91e+06** — alto pero esperado.  
Viene de `loan_to_income_ratio` dominando la regresion. No indica un problema  
tecnico grave — el modelo converge y las predicciones son estables.  
En STATA, este seria el VIF elevado de `loan_to_income_ratio`.

**Jarque-Bera test: p=0.00** — residuos no son normales.  
Curtosis=7.67 y skewness=-0.83 confirman distribucion leptocurtica con cola izquierda.  
El modelo subestima montos grandes sistematicamente. Viola el supuesto 4 de Gauss-Markov  
para inferencia — los p-values son aproximados, no exactos.

**R2 = R2 ajustado = 0.810** — diferencia de 0.0001.  
Las 22 features todas aportan valor real. No hay features puramente ruidosas
(aunque las no significativas tienen efecto estadisticamente indistinguible de cero).

In [ ]:
X_sm = sm.add_constant(X_train)
modelo_sm = sm.OLS(y_train, X_sm).fit()
print(modelo_sm.summary())

In [ ]:
# Variables significativas vs no significativas
pvals  = modelo_sm.pvalues.drop('const')
sig    = pvals[pvals < 0.05].sort_values()
no_sig = pvals[pvals >= 0.05].sort_values()

print(f'Significativas al 5% ({len(sig)}/{len(pvals)}):')
for var, p in sig.items():
    print(f'  {var:<40} beta={modelo_sm.params[var]:>10,.2f}  p={p:.4f}')

print(f'\nNO significativas ({len(no_sig)}/{len(pvals)}):')
for var, p in no_sig.items():
    print(f'  {var:<40} p={p:.4f}')

## Resumen para Respuestas.md

### Pregunta 2.1 — Metricas y evaluacion del modelo

**MAE = $7,561 | RMSE = $11,207 | R² = 0.817**

El modelo funciona bien en terminos de capacidad predictiva: explica el 81.7% de la  
variabilidad del monto del prestamo. Sin embargo, hay limitaciones importantes:

1. **Sin overfitting**: R2_train (0.810) ≈ R2_test (0.817). El modelo generaliza correctamente.

2. **Heterocedasticidad**: el grafico de residuos muestra patron de abanico.  
   La varianza del error no es constante — viola el supuesto OLS.  
   Consecuencia: los errores estandar del summary son aproximados.

3. **Dominancia de loan_to_income_ratio**: este predictor explica la mayor parte  
   del R2 por si solo (r=0.66, coeficiente $41,243 en OLS).  
   Limitacion metodologica: contiene loan_amount en su numerador.

4. **Variables influyentes confirmadas por OLS**:  
   - `loan_to_income_ratio` (p<0.001, beta=$41,243)  
   - `annual_income` (p<0.001, beta=$0.47/USD ingreso)  
   - `age` (p<0.001, beta=$91.21/ano)  

5. **Hallazgo clave**: `credit_score` no es significativo (p=0.525).  
   El score determina la aprobacion (loan_status, excluido) pero no el monto.  
   Conecta con EDA: credit_score correlacionaba con interest_rate (r=-0.49),  
   no directamente con loan_amount (r=+0.11).